In [1]:
pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import pandas as pd

url = "https://prodapi.greatnonprofits.org/front/api/v1/nonprofits"

params = {
    "all": "false",
    "search": "",
    "zipcode": "",
    "country": "",
    "state": "CA",
    "city": "san-diego",
    "category": "",
    "sort_by": "most_recently_reviewed",
    "page": 1,
    "size": 25
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

rows = []


r = requests.get(url, params=params, headers=headers)
data = r.json()

total_pages = data["total_pages"]

print("Total pages:", total_pages)

for page in range(1, total_pages + 1):

    params["page"] = page

    r = requests.get(url, params=params, headers=headers)
    data = r.json()

    for org in data["results"]:

        category = None
        if org["category"]:
            category = org["category"][0]["name"]

        rows.append({
            "name": org["title"],
            "rating": org["rating"],
            "reviews": org["reviews"],
            "category": category,
            "location": org["address"],
            "profile_url": f"https://greatnonprofits.org/org/{org['slug']}"
        })

df = pd.DataFrame(rows)

print("Total nonprofits collected:", len(df))

Total pages: 40
Total nonprofits collected: 1000


In [3]:
df

,name,rating,reviews,category,location,profile_url
0,San Diego Dance Theatre,5,139,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-danc...
1,San Diego Civic Youth Ballet,5,96,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-civi...
2,Jewish Family Service of San Diego,5,48,Children & Youth,"San Diego, CA",https://greatnonprofits.org/org/jewish-family-...
3,"Plant With Purpose (Floresta USA, Inc.)",5,32,Food,"San Diego, CA",https://greatnonprofits.org/org/plant-with-pur...
4,Community HousingWorks (CHW),4,23,Homeless & Housing,"San Diego, CA",https://greatnonprofits.org/org/community-hous...
...,...,...,...,...,...,...
995,Combined Arts & Education Council of San Diego...,0,0,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/combined-arts-...
996,Sigma Delta Epsilon Graduate Women In Science Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/sigma-delta-ep...
997,Law Library Justice Foundation,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/law-library-ju...
998,San Diego Theosophists Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/san-diego-theo...


In [4]:
import numpy as np
np.nan

nan

In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np

def get_contact_info(url):

    headers = {"User-Agent": "Mozilla/5.0"}
    
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    email = np.nan
    phone = np.nan
    website = np.nan
    address = np.nan
    ein = np.nan


    for a in soup.find_all("a", href=True):
        if "mailto:" in a["href"]:
            email = a["href"].replace("mailto:", "")
            break


    for a in soup.find_all("a", href=True):
        if "tel:" in a["href"]:
            phone = a["href"].replace("tel:", "")
            break


    for a in soup.find_all("a", href=True):
        if "http" in a["href"] and "greatnonprofits" not in a["href"]:
            website = a["href"]
            break


    text = soup.get_text()
    if "EIN" in text:
        idx = text.find("EIN")
        ein = text[idx:idx+20]


    if "San Diego" in text:
        idx = text.find("San Diego")
        address = text[idx:idx+80]

    return email, phone, website, address, ein

In [6]:
emails = []
phones = []
websites = []
addresses = []
eins = []

for url in df["profile_url"]:
    
    email, phone, website, address, ein = get_contact_info(url)

    emails.append(email)
    phones.append(phone)
    websites.append(website)
    addresses.append(address)
    eins.append(ein)

df["email"] = emails
df["phone"] = phones
df["website"] = websites
df["address_full"] = addresses
df["ein"] = eins

In [7]:
df

,name,rating,reviews,category,location,profile_url,email,phone,website,address_full,ein
0,San Diego Dance Theatre,5,139,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-danc...,NaN,NaN,NaN,NaN,NaN
1,San Diego Civic Youth Ballet,5,96,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-civi...,NaN,NaN,NaN,NaN,NaN
2,Jewish Family Service of San Diego,5,48,Children & Youth,"San Diego, CA",https://greatnonprofits.org/org/jewish-family-...,NaN,NaN,NaN,NaN,NaN
3,"Plant With Purpose (Floresta USA, Inc.)",5,32,Food,"San Diego, CA",https://greatnonprofits.org/org/plant-with-pur...,NaN,NaN,NaN,NaN,NaN
4,Community HousingWorks (CHW),4,23,Homeless & Housing,"San Diego, CA",https://greatnonprofits.org/org/community-hous...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
995,Combined Arts & Education Council of San Diego...,0,0,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/combined-arts-...,NaN,NaN,NaN,NaN,NaN
996,Sigma Delta Epsilon Graduate Women In Science Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/sigma-delta-ep...,NaN,NaN,NaN,NaN,NaN
997,Law Library Justice Foundation,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/law-library-ju...,NaN,NaN,NaN,NaN,NaN
998,San Diego Theosophists Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/san-diego-theo...,NaN,NaN,NaN,NaN,NaN


In [8]:
df.iloc[0]

name                                      San Diego Dance Theatre
rating                                                          5
reviews                                                       139
category                                           Arts & Culture
location                                            San Diego, CA
profile_url     https://greatnonprofits.org/org/san-diego-danc...
email                                                         NaN
phone                                                         NaN
website                                                       NaN
address_full                                                  NaN
ein                                                           NaN
Name: 0, dtype: object

In [9]:
import requests

url = "https://prodapi.greatnonprofits.org/front/api/v1/nonprofit/san-diego-automotive-museum-inc"

headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)

data = r.json()

print(data["results"])

{'id': 454273, 'title': 'San Diego Automotive Museum Inc', 'logo': None, 'ein': '33-0147359', 'aka': 'SAN DIEGO AUTOMOTIVE MUSEUM', 'slug': 'san-diego-automotive-museum-inc', 'org_email': 'Brandi@sdautomuseum.org', 'causes': [{'title': 'Museums', 'slug': 'museums'}], 'phone': '(619) 231-2886', 'address1': '2080 Pan American Plaza', 'address2': '', 'is_donation_enable': False, 'is_claimed': False, 'zipcode': '92101-1636', 'city': 'San Diego', 'state': 'California', 'state_full': 'California', 'country': 'USA', 'bens_direct': '', 'population': 'youth explore a wide variety of automotive career opportunities through field trips, hands-on demonstrations, and career readiness workshops.  ', 'website': 'sdautomuseum.org', 'programs': 'This innovative program combines career exploration, independent research, and skill-building through project-based hands-on learning to prepare students for internships and other pathways towards successful careers in the automotive industry. The IGNITE progra

In [13]:
df["slug"] = df["profile_url"].str.split("/").str[-1]

In [18]:
import requests
import numpy as np
import time

headers = {"User-Agent": "Mozilla/5.0"}

emails = []
phones = []
websites = []
eins = []

for slug in df["slug"]:

    url = f"https://prodapi.greatnonprofits.org/front/api/v1/nonprofit/{slug}"

    try:
        r = requests.get(url, headers=headers)
        org = r.json().get("results", {})

        emails.append(org.get("org_email") or np.nan)
        phones.append(org.get("phone") or np.nan)
        websites.append(org.get("website") or np.nan)
        eins.append(org.get("ein") or np.nan)

    except:
        emails.append(np.nan)
        phones.append(np.nan)
        websites.append(np.nan)
        eins.append(np.nan)

    time.sleep(0.2)

df["email"] = emails
df["phone"] = phones
df["website"] = websites
df["ein"] = eins

In [21]:
df

,name,rating,reviews,category,location,profile_url,email,phone,website,address_full,ein,slug
0,San Diego Dance Theatre,5,139,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-danc...,katie.sddancetheater@gmail.com,619.225.1803,www.sandiegodancetheater.org,NaN,23-7112940,san-diego-dance-theatre
1,San Diego Civic Youth Ballet,5,96,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/san-diego-civi...,sdcyb@sdcyb.org,619-233-3060,www.sdcyb.org,NaN,23-7079373,san-diego-civic-youth-ballet
2,Jewish Family Service of San Diego,5,48,Children & Youth,"San Diego, CA",https://greatnonprofits.org/org/jewish-family-...,jfsonline@jfssd.org,858-637-3000,www.jfssd.org,NaN,95-1644024,jewish-family-service-of-san-diego
3,"Plant With Purpose (Floresta USA, Inc.)",5,32,Food,"San Diego, CA",https://greatnonprofits.org/org/plant-with-pur...,info@plantwithpurpose.org,858-274-3718,https://www.plantwithpurpose.org,NaN,33-0052976,plant-with-purpose-floresta-usa-inc
4,Community HousingWorks (CHW),4,23,Homeless & Housing,"San Diego, CA",https://greatnonprofits.org/org/community-hous...,socialmedia@chworks.org,888-884-4249,https://chworks.org/,NaN,33-0317950,community-housingworks-chw
...,...,...,...,...,...,...,...,...,...,...,...,...
995,Combined Arts & Education Council of San Diego...,0,0,Arts & Culture,"San Diego, CA",https://greatnonprofits.org/org/combined-arts-...,NaN,NaN,NaN,NaN,95-6115044,combined-arts-education-council-of-san-diego-c...
996,Sigma Delta Epsilon Graduate Women In Science Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/sigma-delta-ep...,NaN,NaN,NaN,NaN,95-6095676,sigma-delta-epsilon-graduate-women-in-science-inc
997,Law Library Justice Foundation,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/law-library-ju...,NaN,NaN,NaN,NaN,95-6048420,law-library-justice-foundation
998,San Diego Theosophists Inc,0,0,None,"San Diego, CA",https://greatnonprofits.org/org/san-diego-theo...,NaN,(760) 765-1090,www.theosophysandiego.org,NaN,95-6049872,san-diego-theosophists-inc


In [33]:
df.to_csv("EIN.csv")